# 05 Final Load Prep

**Purpose:** Extract the final, refined dataset and pre-aggregate specialized tables to be loaded directly into Tableau. This ensures the dashboard runs quickly and contains exactly the structured data it needs.

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'retail_cleaned.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset for final prep: {df.shape}")

### Step 1: Export Monthly Revenue Summary
**Why:** Tableau handles time-series better when data is pre-aggregated, reducing computational load on the BI tool.

In [ ]:
monthly_summary = (
    df.groupby(['transaction_year', 'transaction_month'])
    .agg(total_revenue=('total_spent', 'sum'), total_transactions=('transaction_id', 'count'))
    .reset_index()
    .sort_values(['transaction_year', 'transaction_month'])
)
monthly_summary.to_csv(PROCESSED_DIR / 'monthly_revenue_summary.csv', index=False)
print("Saved: monthly_revenue_summary.csv")
monthly_summary.head(3)

### Step 2: Export Revenue by Category
**Why:** Category performance is a top-level KPI. Exporting this independently guarantees immediate load for bar charts.

In [ ]:
cat_summary = (
    df.groupby('category')
    .agg(total_revenue=('total_spent', 'sum'), total_transactions=('transaction_id', 'count'))
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)
cat_summary.to_csv(PROCESSED_DIR / 'revenue_by_category.csv', index=False)
print("Saved: revenue_by_category.csv")
cat_summary.head(3)

### Step 3: Export Discount Impact Table
**Why:** To definitively show executives if discounts are driving higher AOV or just eroding margin.

In [ ]:
discount_impact = (
    df.groupby('discount_applied')
    .agg(
        total_revenue=('total_spent', 'sum'), 
        total_transactions=('transaction_id', 'count'),
        avg_order_value=('total_spent', 'mean')
    )
    .round(2)
    .reset_index()
)
discount_impact['discount_label'] = discount_impact['discount_applied'].map({0: 'No Discount', 1: 'Discount Applied'})

discount_impact.to_csv(PROCESSED_DIR / 'discount_impact.csv', index=False)
print("Saved: discount_impact.csv")
discount_impact

### Step 4: Export Channel Performance Table
**Why:** To compare the total revenue generation and volume of Online vs In-store channels side-by-side.

In [ ]:
channel_perf = (
    df.groupby('location')
    .agg(total_revenue=('total_spent', 'sum'), total_transactions=('transaction_id', 'count'))
    .sort_values('total_revenue', ascending=False)
    .reset_index()
)
channel_perf.to_csv(PROCESSED_DIR / 'channel_performance.csv', index=False)
print("Saved: channel_performance.csv")
channel_perf

### Final Confirmation
All outputs have been generated and saved to `data/processed/`. The analytics pipeline is now complete.